# Build the `Interactive_Workloads` Direct Lake model

Creates a Direct Lake semantic model over `gold_heavy_interactive_items` and adds
measures for the warehouse-migration priority report (heaviest **interactive**
reports & semantic models, with the interactive-vs-refresh split).

## Prerequisites
- `heavy_interactive_workloads` has run, so `gold_heavy_interactive_items` exists.
- Workspace **XMLA endpoint = Read Write** (for the measure step).

In [ ]:
%pip install -q semantic-link-labs "PyJWT>=2.6.0"

In [ ]:
from sempy_labs.directlake import generate_direct_lake_semantic_model

MODEL  = "Interactive_Workloads"
LH     = "lh_fabric_management"
SCHEMA = "fabricmanagement"
TABLES = {"gold_heavy_interactive_items": f"{SCHEMA}.gold_heavy_interactive_items"}

generate_direct_lake_semantic_model(
    dataset=MODEL, tables=TABLES, source=LH, source_type="Lakehouse",
    overwrite=True, refresh=True,
)
print(f"Direct Lake model '{MODEL}' created over {list(TABLES)}")

In [ ]:
from sempy_labs.tom import connect_semantic_model

T = "gold_heavy_interactive_items"
with connect_semantic_model(dataset="Interactive_Workloads", readonly=False) as tom:
    tom.add_measure(T, "Interactive CU", f"SUM({T}[interactive_cu_s])", format_string="#,0")
    tom.add_measure(T, "Background CU",  f"SUM({T}[background_cu_s])",  format_string="#,0")
    tom.add_measure(T, "Total CU",       f"SUM({T}[total_cu_s])",       format_string="#,0")
    tom.add_measure(T, "% Interactive",  "DIVIDE([Interactive CU], [Total CU])", format_string="0.0%")
    tom.add_measure(T, "Interactive Operations", f"SUM({T}[interactive_ops])", format_string="#,0")
    tom.add_measure(T, "Items", f"DISTINCTCOUNT({T}[item_id])", format_string="#,0")
print("Measures added.")

## Verify — measures

In [ ]:
from sempy_labs.tom import connect_semantic_model
print("Measures on Interactive_Workloads:")
with connect_semantic_model(dataset="Interactive_Workloads", readonly=True) as tom:
    for m in tom.all_measures():
        print(f"  [{m.Parent.Name}]  {m.Name}")

## Build the report (in the Fabric UI)

On the **`Interactive_Workloads`** model → **Auto-create report** (or New report):

1. **Table — warehouse priority list:** `artifact_kind`, `item_name`, `workspace`,
   `capacity_name`, `[Interactive CU]`, `[Total CU]`, `[% Interactive]`, `top_operation`
   — sort by `[Interactive CU]` desc; filter **`artifact_kind` in (Report, Dataset,
   PaginatedReport, Model, Datamart)** for the reports/semantic-models view.
2. **Cards:** `[Interactive CU]`, `[% Interactive]`, `[Items]`.
3. (Optional) **Bar:** `[Interactive CU]` by `capacity_name`, or by `artifact_kind`.
4. **Save as `Interactive Workload Priority`.**

The heavy interactive datasets/reports at the top are your candidates to push into
the warehouse; a low `% Interactive` means the item is refresh-heavy (a different fix).